# Mobile Layout & Bottom Toolbar (`craftMobileBar`)

Mobile-scoped bottom toolbar (`#craft-mbar`), backdrop (`#craft-mobile-backdrop`), and responsive CSS rules under `html.craft-mobile` (activated by `window.craftMobile` in `05_dialog_chrome.ipynb`).

Because `12_mobile.ipynb` sorts last in `nbs/craft/`, its CSS cleanly overrides desktop rules at the end of `dist/CRAFT.css`, and `craftMobileBar` initializes after all core UI, file tree, and editor modules.

In [ ]:
#| export_ft CRAFT.js
#| eval: false
MB_PATHS = dict(
    menu='M4 6h16M4 12h16M4 18h16',
    files='M3 6a1 1 0 0 1 1-1h5l2 2h9a1 1 0 0 1 1 1v10a1 1 0 0 1-1 1H4a1 1 0 0 1-1-1z',
    Code='M8 6l-6 6 6 6m8-12 6 6-6 6M14 4l-4 16',
    Note='M5 3h10l4 4v14H5z M15 3v5h4M8 12h8m-8 4h8',
    Prompt='M3 4h18v12H9l-6 4z M7 10h.01m5 0h.01m5 0h.01',
    Raw='M3 5h18v14H3z M7 9v6',
    tab='M3 12h13m-4-4 4 4-4 4M20 6v12',
    run='M7 5v14l11-7z',
    done='M5 12.5l4.5 4.5L19 7.5',
    down='M12 5v14m-6-6 6 6 6-6',
)
MB_TYPES = ('Code', 'Note', 'Prompt', 'Raw')
mb_btn = lambda act, k, lbl: Button(svg_icon(MB_PATHS[k], 20, 1.7), Span(lbl), type='button', data_act=act, aria_label=lbl)
mb_bulge = lambda edge, d: Button(
    ft('svg', ft('path', klass='craft-edge-stroke', d=d), aria_hidden='true', viewBox='0 0 36 10', width='36', height='10'),
    type='button', klass='craft-edge-bulge', data_edge=edge,
    aria_label='Insert cell above' if edge == 'above' else 'Insert cell below')

TEMPLATES = dict(
    mb_bar=Nav(
        Div(mb_btn('nav', 'menu', 'Menu'), mb_btn('files', 'files', 'Files'), klass='craft-mbar-row', data_row='idle'),
        Div(mb_btn('tab', 'tab', 'Tab'), Span(klass='craft-mbar-spacer'), mb_btn('run', 'run', 'Run'), mb_btn('done', 'done', 'Done'), klass='craft-mbar-row', data_row='edit'),
        id='craft-mbar', aria_label='Mobile toolbar', data_mode='idle'),
    mb_backdrop=craft_overlay(id='craft-mobile-backdrop'),
    mb_jump_btn=Button(svg_icon(MB_PATHS['down'], 18, 1.7), id='craft-scroll-bottom', type='button', aria_label='Jump to bottom'),
    mb_picker=Div(
        Div(klass='craft-type-picker-head craft-edge-picker-head'),
        Div(*[craft_btn(svg_icon(MB_PATHS[t], 17, 1.7), Span(t), role='menuitem', data_type=t) for t in MB_TYPES],
            klass='craft-type-picker-opts craft-edge-picker-opts'),
        id='craft-edge-picker', klass='craft-type-picker', role='menu', aria_label='Insert cell'),
    mb_bulge_above=mb_bulge('above', 'M0 9 C9 9 12 1.5 18 1.5 C24 1.5 27 9 36 9'),
    mb_bulge_below=mb_bulge('below', 'M0 1 C9 1 12 8.5 18 8.5 C24 8.5 27 1 36 1'),
    mb_empty_add=Div(
        Span('Add first cell', klass='craft-empty-add-label'),
        Div(*[craft_btn(svg_icon(MB_PATHS[t], 16, 1.7), Span(t), data_type=t) for t in MB_TYPES],
            klass='craft-empty-add-btns'),
        id='craft-empty-add'),
)

## State, Mode Sync & Cell Creation

Tracks which inline editor owns the soft keyboard, switches `#craft-mbar` between idle pill dock and full-width editing bar, and handles cell creation (including the empty-dialog starter row).

In [ ]:
#| export_js_iife CRAFT.js mobileBar
#| eval: false
window.craftMobileBar?.disable();
  const root = document.documentElement;
  const {mountTpl, qs, qsa, on, htmxPost, patch, rafOnce, onDomChange} = window.__craftCMCore;
  const ac = new AbortController(), sig = {signal: ac.signal}, cap = {capture: true, signal: ac.signal}, pas = {passive: true, signal: ac.signal};
  const bar = mountTpl('mb_bar'), backdrop = mountTpl('mb_backdrop'), jumpBtn = mountTpl('mb_jump_btn');
  const picker = mountTpl('mb_picker'), bulgeAbove = mountTpl('mb_bulge_above'), bulgeBelow = mountTpl('mb_bulge_below');
  const emptyAdd = mountTpl('mb_empty_add');

  const sleep = ms => new Promise(r => setTimeout(r, ms));
  const notify = message => window.UIkit?.notification?.({message, status: 'warning'});
  const dialog = () => document.getElementById('dialog-container');
  const navEl = () => qs('.uk-container > nav.monster-navbar');

  /* Which inline editor (if any) owns the keyboard right now. */
  const editing = () => {
    const c2 = window.craftC2?.active?.();
    if (c2 && window.craftC2.view(c2)?.hasFocus) return {kind: 'code', id: c2, view: window.craftC2.view(c2)};
    const key = window.craftMd?.active?.(), v = key && window.craftMd.view(key);
    if (v?.hasFocus) {
      const [id, part] = key.split(':');
      return {kind: window.craftMd.kind(key), id, part, key, view: v};
    }
    return null;
  };
  const setBarHidden = hidden => {
    const hide = !!hidden && bar.dataset.mode === 'idle' &&
      !root.classList.contains('craft-nav-open') &&
      !window.solveitFileTree?.isOpen?.();
    bar.toggleAttribute('data-hidden', hide);
    jumpBtn.toggleAttribute('data-dock-hidden', hide);
  };
  const syncMode = () => {
    const ed = editing();
    bar.dataset.mode = ed ? 'edit' : 'idle';
    root.classList.toggle('craft-mobile-editing', !!ed);
    if (ed) setBarHidden(false);
    scheduleBulges();
    const run = qs('[data-act="run"]', bar);
    const runnable = ed && (ed.kind === 'code' || (ed.kind === 'prompt' && ed.part !== 'out'));
    run.hidden = !runnable;
    if (runnable) {
      const label = ed.kind === 'prompt' ? 'Send' : 'Run';
      qs('span', run).textContent = label; run.setAttribute('aria-label', label);
    }
    syncJump();
  };
  let modeTimer = null;
  const scheduleMode = () => { clearTimeout(modeTimer); modeTimer = setTimeout(syncMode, 60); };

  async function closeEditors() {
    if (window.craftC2?.active?.()) await window.craftC2.exit();
    if (window.craftMd?.active?.()) await window.craftMd.exit();
  }

  async function createFirst(type) {
    if (type === 'Code') return window.craftC2?.newCodeCell?.();
    const kid = document.getElementById('kid')?.value;
    if (!kid || !window.htmx) return notify('Reload the dialog and try again.');
    const before = new Set(qsa('.editable[id]', dialog()).map(el => el.id));
    await htmxPost('/upsert_msg_', {id_: '', msg_type: type.toLowerCase(), is_input: '1', content: ''});
    for (let i = 0; i < 100; i++) {
      const el = qsa('.editable[id]', dialog()).find(x => !before.has(x.id));
      if (el) { el.scrollIntoView({block: 'nearest'}); window.craftMd?.edit?.(el.id.slice(1), {toEnd: true}); return; }
      await sleep(50);
    }
  }

  let adding = false;
  async function addCell(type, where = 'below', targetCell = null) {
    const container = dialog();
    if (!container || adding || window.craftInsertPreview?.isBusy?.()) return;
    adding = true;
    try {
      const editingCell = document.activeElement?.closest?.('#dialog-container > .editable');
      await closeEditors();
      document.activeElement?.blur?.();
      const cells = qsa(':scope > .editable[id]', container).filter(el => el.getClientRects().length);
      const target = (targetCell?.isConnected && targetCell) || (editingCell?.isConnected && editingCell) || cells.find(el => el.dataset.sm === 'primary') || cells[cells.length - 1];
      if (!target) { await createFirst(type); return; }
      if (!window.craftInsertPreview?.insert) return notify('Reload this dialog to enable inserting cells.');
      await sleep(0);
      window.craftInsertPreview.insert(where === 'above' ? 'above' : 'below', {type, cell: target, force: true});
    } catch (err) {
      console.error('craftMobileBar: add cell failed', err);
      notify('Could not add the cell.');
    } finally { adding = false; }
  }

## Cell-Edge Bulges & Drag-to-Insert Picker

Attaches top/bottom brace-style bulges to the active or selected cell and drives the press-and-drag (or tap) four-type insertion picker.

In [ ]:
#| export_js_iife CRAFT.js mobileBar
#| eval: false
/* ── Cell-edge bulges: attach to the editing cell or primary selected cell ── */
  let bulgeHost = null;
  const detachBulges = () => {
    bulgeHost?.classList.remove('craft-bulge-host');
    bulgeAbove.remove(); bulgeBelow.remove();
    bulgeHost = null;
  };
  const activeEditingCell = cells => {
    const c2 = window.craftC2?.active?.();
    if (c2) {
      const el = document.getElementById('_' + String(c2).replace(/^_/, ''));
      if (el && cells.includes(el)) return el;
    }
    const mdKey = window.craftMd?.active?.();
    if (mdKey) {
      const el = document.getElementById('_' + String(mdKey).split(':')[0].replace(/^_/, ''));
      if (el && cells.includes(el)) return el;
    }
    return cells.find(el => el.dataset.c2Editing === 'true' || el.dataset.mdEditing === 'true') || null;
  };
  const syncBulges = () => {
    if (!mounted) return;
    const container = dialog();
    if (!container) { detachBulges(); emptyAdd.remove(); return; }
    const cells = qsa(':scope > .editable[id]', container).filter(el => !el.hidden && el.getClientRects().length);
    if (!cells.length) {
      detachBulges();
      if (!emptyAdd.isConnected) container.append(emptyAdd);
      return;
    }
    emptyAdd.remove();
    const target = activeEditingCell(cells) || cells.find(el => el.dataset.sm === 'primary') || null;
    if (target !== bulgeHost) {
      detachBulges();
      if (target) {
        target.classList.add('craft-bulge-host');
        target.append(bulgeAbove, bulgeBelow);
        bulgeHost = target;
      }
    }
  };
  const scheduleBulges = rafOnce(syncBulges);

  /* ── Bulge drag-to-insert + tap-to-pick interaction ── */
  let pickerState = null; // {edge, cell, pointerId, startX, startY, dragged, hotType, tapOpen}
  const setHot = type => {
    if (pickerState) pickerState.hotType = type;
    for (const b of qsa('button[data-type]', picker)) {
      b.toggleAttribute('data-hot', b.dataset.type === type);
    }
  };
  const closePicker = () => {
    bulgeAbove.removeAttribute('data-active');
    bulgeBelow.removeAttribute('data-active');
    picker.classList.remove('open');
    setHot(null);
    pickerState = null;
  };
  const positionPicker = (anchorRect, edge) => {
    patch(picker, {dataset: {edge}, classes: {open: true}});
    qs('.craft-edge-picker-head', picker).textContent = edge === 'above' ? 'Insert above' : 'Insert below';
    const pw = picker.offsetWidth || 280;
    const ph = picker.offsetHeight || 76;
    const cx = anchorRect.left + anchorRect.width / 2;
    const left = Math.max(8, Math.min(cx - pw / 2, innerWidth - pw - 8));
    let top;
    if (edge === 'above') {
      const aboveTop = anchorRect.top - ph - 14;
      top = aboveTop >= 12 ? aboveTop : Math.min(anchorRect.bottom + 14, innerHeight - ph - 12);
    } else {
      const belowTop = anchorRect.bottom + 14;
      top = (belowTop + ph <= innerHeight - 12) ? belowTop : Math.max(12, anchorRect.top - ph - 14);
    }
    picker.style.left = Math.round(left) + 'px';
    picker.style.top = Math.round(top) + 'px';
  };
  const hitOption = (x, y) => {
    const PAD = 10;
    for (const b of qsa('button[data-type]', picker)) {
      const r = b.getBoundingClientRect();
      if (x >= r.left - PAD && x <= r.right + PAD && y >= r.top - PAD && y <= r.bottom + PAD) {
        return b.dataset.type;
      }
    }
    return null;
  };
  const selectCell = cell => {
    const container = dialog();
    if (!container || !cell?.isConnected) return;
    for (const el of qsa(':scope > .editable[data-sm="primary"], :scope > .editable[data-sm="secondary"]', container)) {
      if (el !== cell) el.setAttribute('data-sm', 'unselected');
    }
    cell.setAttribute('data-sm', 'primary');
  };
  const onBulgePointerDown = e => {
    const handle = e.currentTarget;
    const cell = handle.closest('#dialog-container > .editable');
    if (!cell) return;
    e.preventDefault(); e.stopPropagation();
    const edge = handle.dataset.edge;
    if (pickerState?.tapOpen && pickerState.edge === edge && pickerState.cell === cell) {
      closePicker();
      return;
    }
    closePicker();
    handle.dataset.active = 'true';
    handle.setPointerCapture?.(e.pointerId);
    pickerState = {edge, cell, pointerId: e.pointerId, startX: e.clientX, startY: e.clientY, dragged: false, hotType: null, tapOpen: false};
    positionPicker(handle.getBoundingClientRect(), edge);
    if (window.craftC2?.active?.() || window.craftMd?.active?.()) {
      selectCell(cell);
      closeEditors().then(() => {
        if (cell.isConnected) selectCell(cell);
        scheduleMode();
      }).catch(err => console.error('craftMobileBar: commit on bulge failed', err));
    }
  };
  const onBulgePointerMove = e => {
    if (!pickerState || e.pointerId !== pickerState.pointerId || pickerState.tapOpen) return;
    const dist = Math.hypot(e.clientX - pickerState.startX, e.clientY - pickerState.startY);
    if (dist > 8) pickerState.dragged = true;
    if (pickerState.dragged) setHot(hitOption(e.clientX, e.clientY));
  };
  const onBulgePointerUp = e => {
    if (!pickerState || e.pointerId !== pickerState.pointerId || pickerState.tapOpen) return;
    const {edge, cell, dragged, hotType} = pickerState;
    if (dragged) {
      closePicker();
      if (hotType) addCell(hotType, edge, cell);
    } else {
      pickerState.tapOpen = true;
      pickerState.pointerId = null;
    }
  };
  const onBulgePointerCancel = e => {
    if (pickerState && e.pointerId === pickerState.pointerId) closePicker();
  };
  const onBulgeClick = e => { e.preventDefault(); e.stopPropagation(); };
  const bulgeEvs = {pointerdown: onBulgePointerDown, pointermove: onBulgePointerMove, pointerup: onBulgePointerUp, pointercancel: onBulgePointerCancel, click: onBulgeClick};
  for (const b of [bulgeAbove, bulgeBelow]) on(b, bulgeEvs, sig);
  const onPickerClick = e => {
    const opt = e.target.closest('button[data-type]');
    if (!opt || !pickerState) return;
    e.preventDefault(); e.stopPropagation();
    const {edge, cell} = pickerState;
    closePicker();
    addCell(opt.dataset.type, edge, cell);
  };
  on(picker, 'click', onPickerClick, sig);
  on(emptyAdd, 'click', e => {
    const opt = e.target.closest('button[data-type]');
    if (!opt) return;
    e.preventDefault(); e.stopPropagation();
    addCell(opt.dataset.type, 'below', null);
  }, sig);

## Editing Toolbar Actions, Follow-Cell & Scroll Chrome

Implements `Tab`, `Run/Send`, and `Done` toolbar actions, keeps running cells scrolled into view while outputs grow, hides the idle dock on scroll-down, and toggles the floating jump-to-bottom button.

In [ ]:
#| export_js_iife CRAFT.js mobileBar
#| eval: false
/* After Run/Send the output lands below the fold: keep the cell in view while it grows,
     until it has been still for ~10s or the user scrolls on their own. */
  let stopFollow = null;
  function followCell(id) {
    stopFollow?.();
    const elId = '_' + String(id).replace(/^_/, '');
    let lastH = -1, quiet = 0;
    let offStop = null;
    const stop = () => {
      clearInterval(timer); clearTimeout(cap);
      offStop?.();
      if (stopFollow === stop) stopFollow = null;
    };
    const tick = () => {
      const el = document.getElementById(elId);
      if (!el) return stop();
      const h = Math.round(el.getBoundingClientRect().height);
      if (h !== lastH) { lastH = h; quiet = 0; el.scrollIntoView({block: 'nearest'}); }
      else if (++quiet > 40) stop();
    };
    const timer = setInterval(tick, 250), cap = setTimeout(stop, 120000);
    offStop = on(window, 'touchstart wheel', stop, {capture: true, passive: true});
    stopFollow = stop;
  }

  /* Run the editor's own Tab binding (accepts completions, indents); plain indent if nothing handled it. */
  function pressTab({view, kind}) {
    const ev = new KeyboardEvent('keydown', {key: 'Tab', code: 'Tab', keyCode: 9, bubbles: true, cancelable: true});
    view.contentDOM.dispatchEvent(ev);
    if (!ev.defaultPrevented) view.dispatch(view.state.replaceSelection(kind === 'code' ? '    ' : '  '));
    view.focus();
  }

  const toggleNav = force => {
    const open = root.classList.toggle('craft-nav-open', force);
    qs('[data-act="nav"]', bar).setAttribute('aria-pressed', String(open));
    if (open) setBarHidden(false);
  };

  const actions = {
    nav: () => toggleNav(),
    files: () => { toggleNav(false); setBarHidden(false); window.solveitFileTree?.toggle(false); },
    tab: () => { const ed = editing(); if (ed) pressTab(ed); },
    run: () => {
      const ed = editing();
      if (!ed) return;
      if (ed.kind === 'code') window.craftC2.run(ed.id, 'run');
      else if (ed.kind === 'prompt' && ed.part !== 'out') window.craftMd.run(ed.id, 'run');
      else return;
      followCell(ed.id);
    },
    done: () => {
      const ed = editing();
      if (!ed) return;
      if (ed.kind === 'code') window.craftC2.exit(ed.id); else window.craftMd.exit(ed.key);
    },
  };
  const onBarMouseDown = e => { if (e.target.closest('button')) e.preventDefault(); };  // keep the editor (and soft keyboard) focused
  const onBarClick = e => {
    const b = e.target.closest('button[data-act]');
    if (!b) return;
    e.preventDefault(); e.stopPropagation();
    Promise.resolve(actions[b.dataset.act]?.(b)).catch(err => console.error('craftMobileBar', err)).finally(scheduleMode);
  };
  on(bar, {mousedown: onBarMouseDown, click: onBarClick}, sig);
  on(backdrop, 'click', () => window.solveitFileTree?.close?.(), sig);

  /* ── Scroll-hiding floating dock & jump-to-bottom button ── */
  let scrollerEl = null, lastScrollY = 0;
  const scrollMetrics = () => {
    const sc = dialog();
    if (sc && sc.scrollHeight > sc.clientHeight + 4) {
      return {el: sc, top: sc.scrollTop, height: sc.scrollHeight, client: sc.clientHeight};
    }
    const doc = document.scrollingElement || document.documentElement;
    return {el: window, top: window.scrollY || doc.scrollTop, height: doc.scrollHeight, client: window.innerHeight};
  };
  const syncJump = () => {
    if (!mounted || bar.dataset.mode === 'edit' || !dialog()) { jumpBtn.classList.remove('visible'); return; }
    const m = scrollMetrics();
    const distFromBottom = m.height - m.top - m.client;
    jumpBtn.classList.toggle('visible', distFromBottom > 360);
  };
  const onScroll = () => {
    if (!mounted) return;
    if (pickerState?.tapOpen) closePicker();
    const m = scrollMetrics();
    const dy = m.top - lastScrollY;
    lastScrollY = m.top;
    const distFromBottom = m.height - m.top - m.client;
    if (m.top < 40 || distFromBottom < 40) setBarHidden(false);
    else if (dy > 14) setBarHidden(true);
    else if (dy < -10) setBarHidden(false);
    syncJump();
  };
  let offScroll = null;
  const bindScroller = () => {
    const next = dialog();
    if (next === scrollerEl) return;
    offScroll?.();
    scrollerEl = next;
    offScroll = on(scrollerEl, 'scroll', onScroll, {passive: true});
    lastScrollY = scrollMetrics().top;
    syncJump();
  };
  on(jumpBtn, 'click', e => {
    e.preventDefault(); e.stopPropagation();
    const m = scrollMetrics();
    (m.el === window ? window : m.el).scrollTo({top: m.height, behavior: 'smooth'});
  }, sig);

## Tap-to-Select, Soft-Keyboard Viewport Lift & Lifecycle

Enforces tap-to-select-first before opening the soft keyboard, lifts `#craft-mbar` above the soft keyboard using `visualViewport`, and exposes `window.craftMobileBar`.

In [ ]:
#| export_js_iife CRAFT.js mobileBar
#| eval: false
/* ── Tap-to-select first on mobile ──
     Tapping an unselected cell (or a different cell while editing another) selects
     it first instead of immediately opening the inline editor and soft keyboard.
     A second tap into the already-selected cell opens the editor normally. */
  let guardedTapCell = null, guardClearTimer = null;
  const INTERACTIVE_SEL = 'button, a, input, textarea, select, summary, label, .craft-edge-bulge, .craft-action-trigger, .craft-table-copy, .craft-omit-marker, .craft-part-restore, .cm-editor';
  const isEditingOther = cell => {
    const c2 = window.craftC2?.active?.();
    if (c2 && ('_' + String(c2).replace(/^_/, '')) !== cell.id) return true;
    const md = window.craftMd?.active?.();
    if (md && ('_' + String(md).split(':')[0].replace(/^_/, '')) !== cell.id) return true;
    return false;
  };
  const onWinMouseDown = e => {
    if (!mounted || e.button !== 0) return;
    const t = e.target;
    if (!(t instanceof Element) || t.closest(INTERACTIVE_SEL)) return;
    const cell = t.closest('#dialog-container > .editable[id]');
    if (!cell) return;
    const editingOther = isEditingOther(cell);
    if (cell.dataset.sm === 'primary' && !editingOther) return;
    guardedTapCell = cell;
    clearTimeout(guardClearTimer);
    guardClearTimer = setTimeout(() => { guardedTapCell = null; }, 800);
    e.stopPropagation();
    selectCell(cell);
    if (editingOther) {
      closeEditors().then(() => {
        if (cell.isConnected) selectCell(cell);
        scheduleMode();
      }).catch(err => console.error('craftMobileBar: close on cell switch failed', err));
    }
  };
  const onWinTapFinish = e => {
    if (!guardedTapCell) return;
    const t = e.target;
    if (t instanceof Element && guardedTapCell.contains(t) && !t.closest(INTERACTIVE_SEL)) {
      e.stopPropagation();
      if (e.type === 'click') {
        e.preventDefault();
        guardedTapCell = null;
      }
    } else if (e.type === 'click') {
      guardedTapCell = null;
    }
  };

  /* Tapping outside the revealed top nav (and outside its dropdowns) hides it again;
     tapping outside a tap-opened edge picker dismisses it. */
  const onDocPointerDown = e => {
    const t = e.target;
    if (pickerState?.tapOpen && !picker.contains(t) && !t.closest?.('.craft-edge-bulge')) closePicker();
    if (!root.classList.contains('craft-nav-open')) return;
    if (bar.contains(t) || navEl()?.contains(t) || t.closest?.('#craft-mobile-toggle, .uk-drop, .uk-dropdown, .uk-modal, [role="dialog"]')) return;
    toggleNav(false);
  };
  on(document, 'pointerdown', onDocPointerDown, cap);

  /* Observe DOM updates for: temp editor auto-close, selected cell changes, scroller changes. */
  const onDomMut = () => {
    if (!mounted) return;
    if (window.craftMobile?.active() && qs('#solveit-temp-editor.open') && window.solveitFileTree?.isOpen?.()) window.solveitFileTree.close();
    bindScroller();
    scheduleBulges();
  };

  /* Soft keyboard: lift the bar above it when the browser only shrinks the visual viewport. */
  /* Only while something editable has focus (the keyboard can't be up otherwise), so a zoomed or oddly sized
     layout viewport never strands the bar mid-screen. With interactive-widget=resizes-content this stays 0. */
  const vv = window.visualViewport;
  const place = () => {
    if (!vv) return;
    const a = document.activeElement;
    const typing = a && a !== document.body && (a.isContentEditable || /^(INPUT|TEXTAREA|SELECT)$/.test(a.tagName));
    const kb = typing ? Math.max(0, Math.round(innerHeight - vv.height - vv.offsetTop)) : 0;
    root.style.setProperty('--craft-kb', (kb > 120 ? kb : 0) + 'px');
  };
  on(vv, 'resize scroll', place, sig);
  on(window, 'resize', place, sig);
  on(window, 'scroll', onScroll, pas);
  on(document, 'focusin focusout', place, cap);
  on(document, 'focusin focusout', scheduleMode, cap);
  const settleEvent = ['htmx', 'after', 'settle'].join(window.htmx?.config?.metaCharacter || ':');
  on(document, settleEvent, () => { scheduleMode(); bindScroller(); scheduleBulges(); }, sig);

  let mounted = false, offMount = null, offDom = null;
  const mount = active => {
    if (active && !mounted) {
      document.body.append(backdrop, bar, jumpBtn, picker);
      offMount = on(window, {mousedown: onWinMouseDown, mouseup: onWinTapFinish, click: onWinTapFinish}, true);
      offDom = onDomChange(onDomMut, {attrs: ['class', 'data-sm', 'data-c2-editing', 'data-md-editing', 'hidden'], delay: 0});
      mounted = true; bindScroller(); syncMode(); place(); syncBulges();
    } else if (!active && mounted) {
      guardedTapCell = null; clearTimeout(guardClearTimer);
      offMount?.(); offMount = null;
      offDom?.(); offDom = null;
      closePicker(); detachBulges(); emptyAdd.remove();
      bar.remove(); backdrop.remove(); jumpBtn.remove(); picker.remove();
      toggleNav(false);
      offScroll?.(); offScroll = null; scrollerEl = null;
      root.classList.remove('craft-mobile-editing');
      mounted = false;
    }
  };
  const start = () => mount(!!window.craftMobile?.active());
  const offChange = window.craftMobile?.onChange(mount);
  if (document.body) start(); else on(document, 'DOMContentLoaded', start, {once: true, signal: ac.signal});

  window.craftMobileBar = {
    addCell,
    disable() {
      offChange?.(); mount(false); clearTimeout(modeTimer); scheduleBulges.cancel();
      ac.abort();
      root.style.removeProperty('--craft-kb');
      delete window.craftMobileBar;
    },
  };

## Mobile CSS: Viewport Guard, Typography & Cell Controls

Prevents horizontal viewport zoom on Android/iOS, tunes touch-friendly spacing, and styles bottom-sheet cell action menus.

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ══ Mobile ══════════════════════════════════════════════════════════════
   Everything below is scoped to `html.craft-mobile`, which CRAFT.js sets for
   touch-only devices (or when toggled manually). Desktop never matches these rules.
   Pieces: readable type scale, floating scroll-hiding pill dock (#craft-mbar),
   top/bottom cell-edge drag-to-insert bulges, bottom-sheet menus, slide-in file
   drawer, tap-to-open top nav (housing the view toggle), and reading ergonomics.
   ──────────────────────────────────────────────────────────────────────── */
#craft-mbar, #craft-scroll-bottom, #craft-empty-add,
.craft-edge-bulge, .solveit-tree-row-more { display: none; }

html.craft-mobile {
  --craft-prose: 14px;
  --craft-prose-lh: 1.45;
  --craft-code: 12.5px;
  --craft-mbar-h: 50px;
  --craft-mbar-space: calc(var(--craft-mbar-h) + env(safe-area-inset-bottom, 0px));
  -webkit-text-size-adjust: 100%;
}
/* Nothing may widen the layout viewport (Android would zoom the whole page out to fit it). */
html.craft-mobile, html.craft-mobile body { overflow-x: clip; max-width: 100%; }
html.craft-mobile .h-screen { height: 100dvh !important; }
/* Reclaim bottom space while reading; reserve full bar height only while editing. */
html.craft-mobile .uk-container { padding-bottom: env(safe-area-inset-bottom, 0px) !important; box-sizing: border-box; }
html.craft-mobile.craft-mobile-editing .uk-container { padding-bottom: var(--craft-mbar-space) !important; }

/* ── Type & spacing ── */
html.craft-mobile .msg-content pre, html.craft-mobile .msg-content code { line-height: 1.4 !important; }
html.craft-mobile .msg-body { padding: 4px 8px !important; }
html.craft-mobile .card-out .msg-body { padding-left: 12px !important; }
html.craft-mobile #dialog-container { padding-top: 12px !important; padding-bottom: 68px !important; }
html.craft-mobile .editable:not(.craft-chain-next) { margin-bottom: 18px !important; }
html.craft-mobile .msg-card[data-mtype="prompt"] > .uk-card-header nav > div.grow > p::before,
html.craft-mobile .msg-card[data-mtype="prompt"] > .uk-card-header [data-tokens]::after { font-size: 12px; }
/* Wide content scrolls inside its cell instead of widening the page. */
html.craft-mobile #dialog-container .editable > .msg-card:not(.collapsed) .msg-content table { display: block; max-width: 100%; overflow-x: auto !important; }
html.craft-mobile #dialog-container .editable > .msg-card:not(.collapsed) .msg-content .craft-table-wrap > table { display: table !important; max-width: none !important; overflow: visible !important; }
html.craft-mobile .craft-table-wrap {
  width: fit-content; max-width: 100%; max-height: 60dvh;
  background-image:
    linear-gradient(to right, var(--craft-bg) 30%, #1f203000),
    linear-gradient(to left, var(--craft-bg) 30%, #1f203000),
    radial-gradient(farthest-side at 0 50%, #0008, #0000),
    radial-gradient(farthest-side at 100% 50%, #0008, #0000);
  background-position: left center, right center, left center, right center;
  background-repeat: no-repeat;
  background-size: 20px 100%, 20px 100%, 12px 100%, 12px 100%;
  background-attachment: local, local, scroll, scroll;
}
html.craft-mobile .msg-content table { font-size: 12.5px !important; }
html.craft-mobile .craft-table-copy { padding: 2px 7px; font-size: 10.5px; }
html.craft-mobile #dialog-container .msg-content :is(img, video, canvas, svg:not([width])) { max-width: 100%; height: auto; }
html.craft-mobile #dialog-container .msg-content :is(iframe, .jp-OutputArea, .output_html) { max-width: 100%; }
html.craft-mobile #dialog-container .editable { min-width: 0; max-width: 100%; }

/* Insert picker on the right edge: replaced by cell-edge bulges, so give the width back. */
html.craft-mobile #craft-insert-preview { display: none !important; }
html.craft-mobile #dialog-container.craft-insert-preview { margin-right: 0 !important; padding-right: 0 !important; }

/* ── Cell controls: quiet until selected, finger-sized bottom-sheet menu ── */
html.craft-mobile .msg-card .craft-action-trigger {
  opacity: .32; pointer-events: auto; width: 40px; height: 30px; flex-basis: 40px; font-size: 22px;
  transition: opacity .15s ease;
}
html.craft-mobile .editable:is([data-sm="primary"], [data-sm="secondary"], :focus-within) .craft-action-trigger,
html.craft-mobile .msg-card .craft-action-trigger[aria-expanded="true"] { opacity: 1; }
html.craft-mobile .craft-menu {
  position: fixed !important;
  left: 0 !important; right: 0 !important; bottom: 0 !important; top: auto !important;
  width: 100% !important; min-width: 0 !important; max-width: 100% !important; max-height: 70dvh; overflow-y: auto;
  margin: 0 !important;
  padding: 10px 12px calc(14px + env(safe-area-inset-bottom, 0px)) !important;
  border: 0 !important; border-top: 1px solid #ffffff1c !important;
  border-radius: 16px 16px 0 0 !important;
  box-shadow: 0 -12px 36px #000b !important;
  box-sizing: border-box;
}
html.craft-mobile .craft-menu::before {
  content: ""; display: block; width: 36px; height: 4px; margin: 0 auto 10px;
  border-radius: 999px; background: #ffffff30;
}
html.craft-mobile .craft-action-menu::backdrop { background: #0009; }
html.craft-mobile .craft-menu button { padding: 12px 14px; font-size: 15px; border-radius: 8px; }
html.craft-mobile .msg-card > .uk-card-header nav > div.grow > p,
html.craft-mobile .msg-card[data-mtype="prompt"] > .uk-card-header nav > div.grow > p > span { opacity: 1; }
html.craft-mobile .craft-omit-marker, html.craft-mobile .craft-part-restore { padding: 6px 10px; font-size: 13px; }
html.craft-mobile .msg-card .uk-card-header :is(button, .uk-btn, a) { min-height: 28px; }

## Mobile CSS: Edge Bulges, Drag Picker, Starter Row & Inline Editors

Styles cell-edge insertion bulges, the floating 4-option picker, the empty-dialog starter row, and soft-wrapping inline CM6 editors.

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── Top/bottom brace-style cell-edge bulges & drag-to-insert picker ── */
html.craft-mobile #dialog-container > .editable.craft-bulge-host {
  position: relative; z-index: 2;
  overflow: visible !important; content-visibility: visible !important; contain: none !important;
}
/* Ensure editing cells also carry the selection border, and punch 34px gaps at
   top-center and bottom-center so the SVG brace curves become part of the border. */
html.craft-mobile #dialog-container > .editable.craft-bulge-host::after {
  content: ""; position: absolute; inset: 0;
  border: 2px solid oklch(62.3% 0.214 259.815); border-radius: 0 !important;
  pointer-events: none; z-index: 100;
  mask-image: linear-gradient(#000, #000), linear-gradient(#000, #000), linear-gradient(#000, #000);
  mask-position: 0 0, center top, center bottom;
  mask-size: 100% 100%, 34px 4px, 34px 4px;
  mask-repeat: no-repeat;
  mask-composite: exclude;
  -webkit-mask-image: linear-gradient(#000, #000), linear-gradient(#000, #000), linear-gradient(#000, #000);
  -webkit-mask-position: 0 0, center top, center bottom;
  -webkit-mask-size: 100% 100%, 34px 4px, 34px 4px;
  -webkit-mask-repeat: no-repeat;
  -webkit-mask-composite: xor;
}
html.craft-mobile .craft-edge-bulge {
  display: flex; align-items: center; justify-content: center;
  position: absolute; left: 50%; z-index: 110;
  width: 56px; height: 28px; padding: 0; border: 0; background: transparent;
  box-sizing: border-box;
  transform: translateX(-50%);
  touch-action: none; user-select: none; -webkit-user-select: none;
  -webkit-tap-highlight-color: transparent; cursor: grab;
}
html.craft-mobile .craft-edge-bulge[data-edge="above"] { top: -14px; align-items: flex-end; padding-bottom: 12px; }
html.craft-mobile .craft-edge-bulge[data-edge="below"] { bottom: -14px; align-items: flex-start; padding-top: 12px; }
html.craft-mobile .craft-edge-bulge svg { display: block; overflow: visible; transition: transform .12s ease; }
html.craft-mobile .craft-edge-bulge .craft-edge-stroke {
  fill: none; stroke: oklch(62.3% 0.214 259.815); stroke-width: 2; stroke-linecap: round;
}
html.craft-mobile .craft-edge-bulge[data-active="true"] .craft-edge-stroke { stroke: var(--craft-green); stroke-width: 2.5; }

/* Mobile geometry overrides for shared #craft-edge-picker (base in 09_cell_insertion) */
html.craft-mobile #craft-edge-picker { z-index: 1050; border-radius: 14px; }
html.craft-mobile #craft-edge-picker .craft-edge-picker-head { font-size: 10.5px; }
html.craft-mobile #craft-edge-picker .craft-edge-picker-opts { align-items: stretch; gap: 6px; }
html.craft-mobile #craft-edge-picker button { flex-direction: column; justify-content: center; gap: 3px; min-width: 60px; padding: 8px 6px; border-radius: 10px; }
html.craft-mobile #craft-edge-picker button:is([data-hot], :active) { transform: scale(1.06); }

/* Empty dialog starter row */
html.craft-mobile #craft-empty-add {
  display: flex; flex-direction: column; align-items: center; gap: 10px;
  margin: 32px auto; padding: 14px; max-width: 340px;
  background: #16161e99; border: 1px dashed #ffffff24; border-radius: 12px;
}
html.craft-mobile #craft-empty-add .craft-empty-add-label { font-size: 12px; color: var(--craft-muted); }
html.craft-mobile #craft-empty-add .craft-empty-add-btns { display: flex; gap: 8px; flex-wrap: wrap; justify-content: center; }
html.craft-mobile #craft-empty-add button { gap: 5px; height: auto; padding: 8px 12px; font-size: 13px; }
html.craft-mobile #craft-empty-add button svg { color: var(--craft-accent); }

/* ── Inline editors (C2 code, notes, prompts, raw) ── */
html.craft-mobile .craft-c2-host .cm-editor,
html.craft-mobile .craft-c2-host .cm-scroller,
html.craft-mobile .craft-c2-host .cm-line,
html.craft-mobile .craft-c2-host .cm-lineNumbers .cm-gutterElement,
html.craft-mobile .craft-md-host.craft-md-mono .cm-editor,
html.craft-mobile .craft-md-host.craft-md-mono .cm-line { line-height: 1.4 !important; }
/* Soft-wrap long lines (CM6 reads white-space from the DOM, so this is a real wrap, not a clip). */
html.craft-mobile .craft-cm-host .cm-content { white-space: pre-wrap !important; overflow-wrap: anywhere; word-break: normal; flex-shrink: 1 !important; min-width: 0; }
html.craft-mobile .craft-c2-host .cm-content, html.craft-mobile .craft-md-host.craft-md-mono .cm-content { padding-block: 6px !important; }
html.craft-mobile :is(.craft-c2-host, .craft-c2-tooltips) .cm-tooltip { font-size: 13px; }
html.craft-mobile :is(.craft-c2-host, .craft-c2-tooltips) .cm-tooltip-autocomplete > ul > li { padding: 6px 10px; }
html.craft-mobile :is(.craft-c2-host, .craft-c2-tooltips) .cm-completionDetail { max-width: 40vw; }
html.craft-mobile .craft-cm-host .cm-search :is(input, button) { font-size: 14px; padding: 6px 8px; }
html.craft-mobile .craft-c2-ai-input { font-size: 16px; padding: 8px; }

## Mobile CSS: Bottom Toolbar, Top Nav, File-Tree Drawer & Overlays

Styles the floating pill dock, editing bar above the soft keyboard, jump-to-bottom button, revealed top nav, slide-over file tree drawer, and mobile modals.

In [ ]:
#| export_css CRAFT.css
#| eval: false
/* ── Bottom toolbar: floating scroll-hiding pill when idle, full-width bar when editing ── */
html.craft-mobile #craft-mbar {
  display: block; position: fixed; z-index: 990;
  box-sizing: border-box;
  user-select: none; -webkit-user-select: none;
  touch-action: manipulation;
}
html.craft-mobile #craft-mbar[data-mode="idle"] {
  left: 50%; right: auto; bottom: calc(12px + env(safe-area-inset-bottom, 0px));
  width: fit-content; height: auto; padding: 4px 6px;
  border-radius: 999px;
  background: #16161ee6; backdrop-filter: blur(12px); -webkit-backdrop-filter: blur(12px);
  border: 1px solid #ffffff1c; box-shadow: 0 8px 28px #000a;
  transform: translateX(-50%);
  transition: transform .2s ease, opacity .18s ease;
}
html.craft-mobile #craft-mbar[data-mode="idle"][data-hidden] {
  transform: translate(-50%, calc(100% + 28px));
  opacity: 0; pointer-events: none;
}
html.craft-mobile #craft-mbar[data-mode="edit"] {
  left: 0; right: 0; bottom: 0;
  height: var(--craft-mbar-space);
  padding: 0 4px env(safe-area-inset-bottom, 0px);
  background: var(--craft-recessed); border-top: 1px solid #ffffff14;
  transform: translateY(calc(-1 * var(--craft-kb, 0px)));
}
html.craft-mobile #craft-mbar .craft-mbar-row { display: none; align-items: stretch; gap: 4px; }
html.craft-mobile #craft-mbar[data-mode="idle"] .craft-mbar-row[data-row="idle"] { display: flex; }
html.craft-mobile #craft-mbar[data-mode="edit"] .craft-mbar-row[data-row="edit"] { display: flex; height: var(--craft-mbar-h); gap: 2px; }
html.craft-mobile #craft-mbar .craft-mbar-spacer { flex: 1 1 auto; }
html.craft-mobile #craft-mbar button {
  display: flex; align-items: center; justify-content: center;
  border: 0; background: transparent; color: #a9b1d6;
  font: 500 11.5px/1 var(--craft-prose-family); letter-spacing: .01em;
  -webkit-tap-highlight-color: transparent; cursor: pointer;
}
html.craft-mobile #craft-mbar .craft-mbar-row[data-row="idle"] button {
  flex: 0 0 auto; flex-direction: row; gap: 6px;
  height: 38px; padding: 0 14px; border-radius: 999px;
}
html.craft-mobile #craft-mbar .craft-mbar-row[data-row="edit"] button {
  flex: 0 0 76px; flex-direction: column; gap: 2px;
  margin: 4px 0; padding: 0; border-radius: 8px; font-size: 10.5px;
}
html.craft-mobile #craft-mbar button[hidden] { display: none; }
html.craft-mobile #craft-mbar button span { overflow: hidden; text-overflow: ellipsis; white-space: nowrap; max-width: 100%; }
html.craft-mobile #craft-mbar button:active { background: var(--craft-hover); color: var(--craft-fg-strong); }
html.craft-mobile #craft-mbar button[aria-pressed="true"],
html.craft-mobile body:has(#solveit-file-tree-panel.open) #craft-mbar button[data-act="files"] { color: var(--craft-accent); background: #7aa2f71c; }
html.craft-mobile #craft-mbar button[data-act="run"] { background: #7aa2f726; color: var(--craft-accent); }
html.craft-mobile #craft-mbar button[data-act="done"] { color: var(--craft-green); }

/* ── Jump-to-bottom floating button ── */
html.craft-mobile #craft-scroll-bottom {
  display: grid; place-items: center;
  position: fixed; right: 14px; bottom: calc(14px + env(safe-area-inset-bottom, 0px)); z-index: 985;
  width: 40px; height: 40px; padding: 0; border-radius: 999px;
  background: #16161ee6; backdrop-filter: blur(10px); -webkit-backdrop-filter: blur(10px);
  border: 1px solid #ffffff22; color: var(--craft-fg); box-shadow: 0 6px 20px #000a;
  opacity: 0; pointer-events: none; transform: translateY(12px) scale(.92);
  transition: opacity .18s ease, transform .18s ease;
  -webkit-tap-highlight-color: transparent; cursor: pointer;
}
html.craft-mobile #craft-scroll-bottom.visible { opacity: 1; pointer-events: auto; transform: none; }
html.craft-mobile #craft-scroll-bottom:active { background: var(--craft-hover); color: var(--craft-accent); }

/* ── Top nav: hidden until Menu, then a solid wrapping bar housing the view toggle ── */
html.craft-mobile .uk-container > nav.monster-navbar {
  position: fixed !important; top: 0; left: 0; right: 0; z-index: 960;
  height: auto !important; min-height: 48px !important;
  display: flex !important; flex-wrap: wrap !important; align-items: center; gap: 6px 14px;
  padding: calc(6px + env(safe-area-inset-top, 0px)) 54px 6px 12px !important;
  background: var(--craft-recessed) !important; border-bottom: 1px solid #ffffff14 !important;
  box-shadow: 0 8px 24px #0009;
  opacity: 0; pointer-events: none; transform: translateY(-100%);
  transition: opacity .16s ease, transform .16s ease;
}
html.craft-mobile.craft-nav-open .uk-container > nav.monster-navbar,
html.craft-mobile .uk-container > nav.monster-navbar:focus-within { opacity: 1; pointer-events: auto; transform: none; }
html.craft-mobile.craft-nav-open .uk-container > nav.monster-navbar > * { pointer-events: auto; }
/* The title block is already hidden and Solveit's own ☰ duplicates ours: drop both so the controls start on row one. */
html.craft-mobile .uk-container > nav.monster-navbar > div.min-w-0,
html.craft-mobile .uk-container > nav.monster-navbar > button { display: none !important; }
html.craft-mobile .uk-container > nav.monster-navbar > div.ml-auto { display: flex !important; flex-wrap: wrap; align-items: center; gap: 6px 12px; width: 100%; margin: 0 !important; padding: 0 !important; }
html.craft-mobile .uk-container > nav.monster-navbar > div.ml-auto > div { flex: 1 1 100%; min-width: 0; justify-content: flex-start !important; gap: 6px 8px !important; }
html.craft-mobile .uk-container > nav.monster-navbar .uk-btn-group { flex-wrap: wrap !important; max-width: 100%; }
html.craft-mobile .uk-container > nav.monster-navbar :is(.uk-btn, label, button, select) { min-height: 36px; min-width: 36px; font-size: 14px !important; }
html.craft-mobile .uk-container > nav.monster-navbar svg,
html.craft-mobile .uk-container > nav.monster-navbar select,
html.craft-mobile .uk-container > nav.monster-navbar h2 { filter: none; }

/* Move the desktop/mobile toggle into the revealed top nav instead of floating over cells. */
html.craft-mobile #craft-mobile-toggle {
  top: calc(8px + env(safe-area-inset-top, 0px)) !important;
  left: auto !important; right: 10px !important; z-index: 970 !important;
  width: 36px !important; height: 36px !important; border-radius: 8px !important;
  background: #242638 !important; border: 1px solid #ffffff28 !important;
  color: var(--craft-fg) !important;
  opacity: 0 !important; pointer-events: none !important; transform: translateY(-100%);
  transition: opacity .16s ease, transform .16s ease;
}
html.craft-mobile.craft-nav-open #craft-mobile-toggle {
  opacity: 1 !important; pointer-events: auto !important; transform: none;
}

/* ── File tree as a drawer ── */
html.craft-mobile #solveit-file-tree-panel {
  position: fixed !important; top: 0; left: 0; bottom: 0;
  z-index: 980;
  flex: none !important; height: auto !important;
  width: min(86vw, 380px) !important; min-width: 0 !important; max-width: none !important;
  padding: calc(12px + env(safe-area-inset-top, 0px)) 10px calc(72px + env(safe-area-inset-bottom, 0px)) 10px !important;
  resize: none; overflow-y: auto; overscroll-behavior: contain;
  transform: translateX(-105%); transition: transform .2s ease, visibility .2s;
  box-shadow: none;
}
html.craft-mobile #solveit-file-tree-panel.open { transform: none; box-shadow: 8px 0 32px #000a; }
html.craft-mobile body:has(#solveit-file-tree-panel.open) #craft-mobile-backdrop { display: block; z-index: 970; background: #0009; }
html.craft-mobile #solveit-file-tree-panel > header { flex-wrap: wrap; gap: 2px; }
html.craft-mobile #solveit-file-tree-panel > header strong { font-size: 15px; margin-right: auto; }
html.craft-mobile #solveit-file-tree-panel > header button { min-width: 40px; min-height: 40px; margin-left: 0; font-size: 1.35rem; border-radius: 6px; }
html.craft-mobile #solveit-file-tree-panel > header button:active { background: var(--craft-hover); }
html.craft-mobile #solveit-file-tree-panel .solveit-tree-shortcut,
html.craft-mobile #solveit-file-tree-panel .solveit-tree-rootbar-hint { display: none; }
html.craft-mobile #solveit-file-tree-panel .solveit-tree-appname { font-size: 15px; }
html.craft-mobile #solveit-file-tree-panel h2#dlg-name { font-size: 14px !important; }
html.craft-mobile #solveit-file-tree-panel .solveit-tree-filter { font-size: 16px; padding: 10px 12px; }
html.craft-mobile #solveit-file-tree-panel .solveit-tree-status { font-size: 13px; }
html.craft-mobile #solveit-file-tree-panel .solveit-tree-rootbar { min-height: 40px; padding: 4px 4px 4px 10px; font-size: 14px; -webkit-touch-callout: none; user-select: none; -webkit-user-select: none; }
html.craft-mobile #solveit-file-tree-panel .custom-treeview-wrapper li { margin: 0; font-size: 15px; }
html.craft-mobile #solveit-file-tree-panel .custom-treeview-wrapper ul ul { padding-left: 16px; }
html.craft-mobile #solveit-file-tree-panel .custom-treeview-wrapper .treeview-node-content {
  min-height: 42px; padding: 2px 2px 2px 6px; box-sizing: border-box;
  -webkit-touch-callout: none; user-select: none; -webkit-user-select: none; -webkit-tap-highlight-color: transparent;
}
html.craft-mobile #solveit-file-tree-panel .custom-treeview-wrapper .treeview-node-content:hover { background: transparent; }
html.craft-mobile #solveit-file-tree-panel .custom-treeview-wrapper .treeview-node-content:active { background: #222434; }
html.craft-mobile #solveit-file-tree-panel img.solveit-tree-icon { width: 20px !important; height: 20px !important; max-width: 20px !important; max-height: 20px !important; flex-basis: 20px; margin-right: 10px; }
html.craft-mobile #solveit-file-tree-panel .solveit-tree-row-upload { width: 36px; height: 36px; opacity: 1; pointer-events: auto; }
html.craft-mobile #solveit-file-tree-panel .solveit-tree-row-more {
  display: inline-flex; flex: 0 0 auto; align-items: center; justify-content: center;
  width: 40px; height: 36px; margin-left: 2px; padding: 0; border: 0; border-radius: 6px;
  background: transparent; color: #cfc9c2aa; font-size: 20px; line-height: 1; cursor: pointer;
}
html.craft-mobile #solveit-file-tree-panel .solveit-tree-rootbar .solveit-tree-row-upload { margin-left: auto; }
html.craft-mobile #solveit-file-tree-panel .solveit-tree-row-more:active,
html.craft-mobile #solveit-file-tree-panel .solveit-tree-row-upload:active { background: #30364d; color: var(--craft-accent); }
html.craft-mobile #solveit-file-tree { padding-bottom: 24px; }
html.craft-mobile #solveit-file-tree .solveit-tree-create input { font-size: 16px; padding: 8px 10px; }
html.craft-mobile #solveit-tree-create-error:not(:empty) { font-size: 13px; }
html.craft-mobile .solveit-tree-upload-pop { width: min(320px, calc(100vw - 24px)); padding: 12px; font-size: 14px; }
html.craft-mobile .solveit-tree-upload-dest { font-size: 13px; }
html.craft-mobile .solveit-tree-upload-x { font-size: 22px; padding: 4px 8px; }
html.craft-mobile .solveit-tree-upload-drop { padding: 16px 10px; font-size: 13px; }
html.craft-mobile .solveit-tree-upload-actions button { padding: 11px 8px; font-size: 14px; }

/* ── Overlays: file editor, launcher, shortcuts ── */
html.craft-mobile #solveit-temp-editor { padding-top: env(safe-area-inset-top, 0px); }
html.craft-mobile #solveit-temp-editor .solveit-temp-editor-bar { flex-basis: 48px; }
html.craft-mobile #solveit-temp-editor .solveit-temp-editor-bar strong { min-width: 0; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; font-size: 14px; }
html.craft-mobile #solveit-temp-editor .solveit-temp-editor-bar button { min-width: 44px; min-height: 44px; font-size: 1.8rem; }
html.craft-mobile #solveit-launcher { padding-top: calc(10px + env(safe-area-inset-top, 0px)); }
html.craft-mobile #solveit-launcher .solveit-launcher-card { width: calc(100vw - 16px); }
html.craft-mobile #solveit-launcher input { font-size: 16px; padding: 14px; }
html.craft-mobile #solveit-launcher ul { max-height: 60dvh; }
html.craft-mobile #solveit-launcher li { padding: 11px 12px; font-size: 14px; }
html.craft-mobile #solveit-launcher li .hint { display: none; }
html.craft-mobile .hk-overlay { padding: 8px; }

Verify `12_mobile.ipynb` exports both `CRAFT.js` and `CRAFT.css`.

In [ ]:
from fastcore.test import test_eq
from craft_dev.auth import REPO_ROOT
from craft_dev.build import extract_nb_assets

assets = extract_nb_assets(REPO_ROOT/'nbs/craft/12_mobile.ipynb')
test_eq(sorted(assets), ['CRAFT.css', 'CRAFT.js'])
js = '\n'.join(code for _, code in assets['CRAFT.js'])
assert 'window.craftMobileBar' in js and '__craftTpl' in js
assert 'html.craft-mobile' in assets['CRAFT.css'][0][1]

## Live `fastcdp` Mobile Verification

In [ ]:
#| live
import json
from fastcdp import Rungs
from craft_dev.cdp import click_at, open_solveit

async with open_solveit(mobile=True) as (page, _, page_errors):
    rungs = Rungs(page)
    assert not page_errors, f"Uncaught page errors in mobile mode: {page_errors}"

    async with rungs('mobile toolbar idle state and nav toggle'):
        assert await page.eval("window.craftMobile?.active() === true && document.documentElement.classList.contains('craft-mobile')")
        assert await page.sel_exists('#craft-mbar[data-mode="idle"]')
        assert await page.sel_count('#craft-mbar button[data-act="add"], #craft-mbar button[data-act="composer"]') == 0
        assert await page.sel_count('#craft-mbar .craft-mbar-row[data-row="idle"] button') == 2
        assert await page.eval("getComputedStyle(document.getElementById('craft-mobile-toggle')).pointerEvents === 'none'")

        await page.sel_click('#craft-mbar button[data-act="nav"]')
        assert await page.eval("document.documentElement.classList.contains('craft-nav-open')")
        assert await page.eval("getComputedStyle(document.getElementById('craft-mobile-toggle')).pointerEvents === 'auto'")
        await page.sel_click('#craft-mbar button[data-act="nav"]')
        assert await page.eval("!document.documentElement.classList.contains('craft-nav-open')")

    async with rungs('selected cell edge bulges and insert picker'):
        has_cell = await page.eval(r"""(() => {
          const el = document.querySelector('#dialog-container > .editable[id]');
          if (!el) return false;
          el.setAttribute('data-sm', 'primary');
          return true;
        })()""")
        if has_cell:
            await page.wait_for_selector('.craft-edge-bulge[data-edge="above"]', timeout=3)
            assert await page.sel_exists('.craft-edge-bulge[data-edge="below"]')
            await page.sel_click('.craft-edge-bulge[data-edge="below"]')
            await page.wait_for_selector('#craft-edge-picker.open', timeout=3)
            assert await page.sel_count('#craft-edge-picker button[data-type]') == 4
            await click_at(page, 10, 10)
            await page.wait_for_selector('#craft-edge-picker.open', present=False, timeout=3)

    async with rungs('mobile file tree drawer and backdrop dismiss'):
        await page.sel_click('#craft-mbar button[data-act="files"]')
        await page.wait_for_selector('#solveit-file-tree-panel.open', timeout=3)
        await page.wait_for_selector('#solveit-file-tree-panel .solveit-tree-row-more', timeout=5)
        await click_at(page, 370, 200)
        await page.wait_for_selector('#solveit-file-tree-panel.open', present=False, timeout=3)

    async with rungs('mobile note edit mode and Done exit'):
        note_id = await page.eval(r"""(() => {
          const el = [...document.querySelectorAll('#dialog-container > .editable[id]')]
            .find(e => (e.querySelector('.card-in')?.dataset?.mtype || e.dataset?.mtype) === 'note');
          return el ? el.id.replace(/^_/, '') : null;
        })()""")
        if note_id:
            await page.eval(f"window.craftMd.edit({json.dumps(note_id)})")
            try:
                await page.wait_for_selector(f'#_{note_id} .cm-editor .cm-content', timeout=15)
                await page.sel_click(f'#_{note_id} .cm-editor .cm-content')
                await page.wait_for_selector('#craft-mbar[data-mode="edit"]', timeout=3)
                await page.sel_click('#craft-mbar button[data-act="done"]')
                await page.wait_for_selector(f'#_{note_id} .cm-editor', present=False, timeout=5)
                await page.wait_for_selector('#craft-mbar[data-mode="idle"]', timeout=3)
            finally:
                if await page.eval("!!window.craftMd?.active?.()"): await page.eval("window.craftMd.exit()")

    assert not page_errors, f"Uncaught page errors in mobile mode: {page_errors}"